### Токенизация

- Обучение токенизаторов BPE, WordPiece и Unigram и сравнение метрик токенизации
- Сравнение результатов токенизации текста разными токенизаторами

In [1]:
import time
import json
from collections import Counter
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tokenizers import (
    Tokenizer,
    models,
    trainers,
    pre_tokenizers,
    decoders,
    processors,
    normalizers,
)
import warnings
warnings.filterwarnings('ignore')

## Обучение токенизаторов

In [ ]:
class TokenizerComparator:
    """
    Класс для сравнения BPE, WordPiece и Unigram токенизаторов
    """
    
    def __init__(self, vocab_size: int = 1000):
        """
        Инициализация компаратора
        
        Args:
            vocab_size: размер словаря для всех токенизаторов
        """
        self.vocab_size = vocab_size
        self.tokenizers = {}
        self.results = {}
    
    def create_bpe_tokenizer(self) -> Tokenizer:
        """Создание BPE токенизатора"""
        tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
        
        # Настройка нормализации
        tokenizer.normalizer = normalizers.Sequence([
            normalizers.NFD(),
            normalizers.Lowercase(),
            normalizers.StripAccents()
        ])
        
        # Настройка предварительной токенизации
        tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
        
        # Настройка декодера
        tokenizer.decoder = decoders.ByteLevel()
        
        # Постобработка
        tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
        
        return tokenizer
    
    def create_wordpiece_tokenizer(self) -> Tokenizer:
        """Создание WordPiece токенизатора"""
        tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
        
        # Настройка нормализации
        tokenizer.normalizer = normalizers.Sequence([
            normalizers.NFD(),
            normalizers.Lowercase(),
            normalizers.StripAccents()
        ])
        
        # Настройка предварительной токенизации
        tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
        
        # Настройка декодера
        tokenizer.decoder = decoders.WordPiece()
        
        return tokenizer
    
    def create_unigram_tokenizer(self) -> Tokenizer:
        """Создание Unigram токенизатора"""
        tokenizer = Tokenizer(models.Unigram())
        
        # Настройка нормализации
        tokenizer.normalizer = normalizers.Sequence([
            normalizers.NFD(),
            normalizers.Lowercase(),
            normalizers.StripAccents()
        ])
        
        # Настройка предварительной токенизации
        tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
        
        # Настройка декодера
        tokenizer.decoder = decoders.Metaspace()
        
        return tokenizer
    
    def train_tokenizers(self, data_path: str):
        """
        Обучение всех трех токенизаторов
        
        Args:
            data_path: путь к файлу с обучающими данными
        """
        print("Начинаю обучение токенизаторов...")
        
        # Создание токенизаторов
        bpe = self.create_bpe_tokenizer()
        wordpiece = self.create_wordpiece_tokenizer()
        unigram = self.create_unigram_tokenizer()
        
        # Настройка тренеров
        bpe_trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
            min_frequency=2
        )
        
        wp_trainer = trainers.WordPieceTrainer(
            vocab_size=self.vocab_size,
            special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
            min_frequency=2
        )
        
        unigram_trainer = trainers.UnigramTrainer(
            vocab_size=self.vocab_size,
            special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
            unk_token="[UNK]"
        )
        
        # Обучение токенизаторов
        print("Обучение BPE токенизатора...")
        start_time = time.time()
        bpe.train([data_path], bpe_trainer)
        bpe_time = time.time() - start_time
        
        print("Обучение WordPiece токенизатора...")
        start_time = time.time()
        wordpiece.train([data_path], wp_trainer)
        wp_time = time.time() - start_time
        
        print("Обучение Unigram токенизатора...")
        start_time = time.time()
        unigram.train([data_path], unigram_trainer)
        unigram_time = time.time() - start_time
        
        self.tokenizers = {
            'BPE': bpe,
            'WordPiece': wordpiece,
            'Unigram': unigram
        }
        
        self.training_times = {
            'BPE': bpe_time,
            'WordPiece': wp_time,
            'Unigram': unigram_time
        }
        
        print("Обучение завершено!")
        
    def analyze_tokenization(self, text_path: str) -> Dict:
        """
        Анализ токенизации для списка текстов
        
        Args:
            texts: список текстов для анализа
            
        Returns:
            словарь с результатами анализа
        """
        results = {
            'BPE': {'tokens': [], 'counts': Counter(), 'encoded': []},
            'WordPiece': {'tokens': [], 'counts': Counter(), 'encoded': []},
            'Unigram': {'tokens': [], 'counts': Counter(), 'encoded': []}
        }

        with open(text_path, 'r', encoding='utf-8') as file:
            text = file.read()
        
        for name, tokenizer in self.tokenizers.items():
            # for text in texts:
                # Токенизация
            encoding = tokenizer.encode(text)
            tokens = encoding.tokens
            
            # Сохранение результатов
            results[name]['tokens'].extend(tokens)
            results[name]['counts'].update(tokens)
            results[name]['encoded'].append({
                'text': text,
                'tokens': tokens,
                'ids': encoding.ids,
                'length': len(tokens)
            })
        
        self.results = results
        return results
    
    def calculate_metrics(self, text_path: str) -> pd.DataFrame:
        """
        Расчет метрик для сравнения токенизаторов
        
        Args:
            texts: список текстов для оценки
            
        Returns:
            DataFrame с метриками
        """
        metrics = []

        with open(text_path, 'r', encoding='utf-8') as file:
            text = file.read()
        
        for name, tokenizer in self.tokenizers.items():
            total_tokens = 0
            unique_tokens = set()
            total_time = 0
            total_chars = 0
            compression_ratios = []
            
            # for text in texts:
                # Измерение времени токенизации
            start_time = time.time()
            encoding = tokenizer.encode(text)
            end_time = time.time()
            
            tokens = encoding.tokens
            total_tokens += len(tokens)
            unique_tokens.update(tokens)
            total_time += (end_time - start_time)
            total_chars += len(text)
            
            # Коэффициент сжатия (символы / токены)
            if len(tokens) > 0:
                compression_ratios.append(len(text) / len(tokens))
            
            # Расчет метрик
            metrics.append({
                'Tokenizer': name,
                'Total Tokens': total_tokens,
                'Unique Tokens': len(unique_tokens),
                'Vocab Utilization': len(unique_tokens) / self.vocab_size * 100,
                'Avg Compression Ratio': sum(compression_ratios) / len(compression_ratios),
                'Training Time (s)': self.training_times[name]
            })
        
        return pd.DataFrame(metrics)
    
    def compare_tokenizations(self, text: str) -> pd.DataFrame:
        """
        Сравнение токенизации одного текста
        
        Args:
            text: текст для токенизации
            
        Returns:
            DataFrame с результатами токенизации
        """
        comparisons = []

        
        for name, tokenizer in self.tokenizers.items():
            encoding = tokenizer.encode(text)
            comparisons.append({
                'Tokenizer': name,
                'Tokens': encoding.tokens,
                'Token IDs': encoding.ids,
                'Token Count': len(encoding.tokens),
                'Decoded Text': tokenizer.decode(encoding.ids)
            })
        
        return pd.DataFrame(comparisons)
    
    def visualize_results(self, metrics_df: pd.DataFrame):
        """
        Визуализация результатов сравнения
        
        Args:
            metrics_df: DataFrame с метриками
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # График 1: Общее количество токенов
        axes[0, 0].bar(metrics_df['Tokenizer'], metrics_df['Total Tokens'])
        axes[0, 0].set_title('Total Tokens Generated')
        axes[0, 0].set_ylabel('Count')
        
        # График 2: Уникальные токены
        axes[0, 1].bar(metrics_df['Tokenizer'], metrics_df['Unique Tokens'], color='green')
        axes[0, 1].set_title('Unique Tokens Used')
        axes[0, 1].set_ylabel('Count')
        
        # График 3: Среднее количество токенов на текст
        axes[0, 2].bar(metrics_df['Tokenizer'], metrics_df['Avg Tokens per Text'], color='orange')
        axes[0, 2].set_title('Average Tokens per Text')
        axes[0, 2].set_ylabel('Count')
        
        # График 4: Использование словаря
        axes[1, 0].bar(metrics_df['Tokenizer'], metrics_df['Vocab Utilization'], color='red')
        axes[1, 0].set_title('Vocabulary Utilization (%)')
        axes[1, 0].set_ylabel('Percentage')
        
        # График 5: Среднее время токенизации
        axes[1, 1].bar(metrics_df['Tokenizer'], metrics_df['Avg Time (ms)'], color='purple')
        axes[1, 1].set_title('Average Tokenization Time (ms)')
        axes[1, 1].set_ylabel('Time (ms)')
        
        # График 6: Коэффициент сжатия
        axes[1, 2].bar(metrics_df['Tokenizer'], metrics_df['Avg Compression Ratio'], color='brown')
        axes[1, 2].set_title('Average Compression Ratio')
        axes[1, 2].set_ylabel('Ratio (chars/tokens)')
        
        plt.tight_layout()
        plt.show()
    
    def plot_token_distribution(self, top_n: int = 20):
        """
        Визуализация распределения токенов
        
        Args:
            top_n: количество наиболее частых токенов для отображения
        """
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for idx, (name, result) in enumerate(self.results.items()):
            most_common = result['counts'].most_common(top_n)
            tokens, counts = zip(*most_common)
            
            axes[idx].barh(tokens, counts)
            axes[idx].set_title(f'Top {top_n} tokens - {name}')
            axes[idx].set_xlabel('Frequency')
            axes[idx].invert_yaxis()
        
        plt.tight_layout()
        plt.show()



In [5]:
comparator = TokenizerComparator(vocab_size=1000)

train_data = '../data/MaP.txt'
    
# Обучение токенизаторов
comparator.train_tokenizers(train_data)

# Анализ токенизации
comparator.analyze_tokenization(train_data)

# Расчет метрик
metrics_df = comparator.calculate_metrics(train_data)

# Вывод результатов
print("\n" + "="*60)
print("МЕТРИКИ СРАВНЕНИЯ ТОКЕНИЗАТОРОВ")
print("="*60)
print(metrics_df.to_string(index=False))

comparator.tokenizers['BPE'].save("../models/tokenizers/bpe_tokenizer.json")
print("\nBPE токенизатор сохранен в 'models/tokenizers/bpe_tokenizer.json'")

comparator.tokenizers['WordPiece'].save("../models/tokenizers/WordPiece_tokenizer.json")
print("\nWordPiece токенизатор сохранен в 'models/tokenizers/WordPiece_tokenizer.json'")

comparator.tokenizers['Unigram'].save("../models/tokenizers/Unigram_tokenizer.json")
print("\nUnigram токенизатор сохранен в 'models/tokenizers/Unigram_tokenizer.json'")



Начинаю обучение токенизаторов...
Обучение BPE токенизатора...
Обучение WordPiece токенизатора...
Обучение Unigram токенизатора...
Обучение завершено!

МЕТРИКИ СРАВНЕНИЯ ТОКЕНИЗАТОРОВ
Tokenizer  Total Tokens  Unique Tokens  Vocab Utilization  Avg Compression Ratio  Training Time (s)
      BPE       1226863            957               95.7               2.433795           3.760382
WordPiece       1186883            983               98.3               2.515777           1.381136
  Unigram       1232667            989               98.9               2.422335          40.999482

BPE токенизатор сохранен в 'models/tokenizers/bpe_tokenizer.json'

WordPiece токенизатор сохранен в 'models/tokenizers/WordPiece_tokenizer.json'

Unigram токенизатор сохранен в 'models/tokenizers/Unigram_tokenizer.json'


## Сравнение токенизации текста

In [4]:
tokenizers = {
    'BPE': Tokenizer.from_file('../models/tokenizers/bpe_tokenizer.json'),
    'WordPiece': Tokenizer.from_file('../models/tokenizers/WordPiece_tokenizer.json'),
    'Unigram': Tokenizer.from_file('../models/tokenizers/unigram_tokenizer.json')
}

In [ ]:
example_text = "Мой дядя самых честных правил"

for name, tokenizer in tokenizers.items():
    encoding = tokenizer.encode(example_text)

    print(f"\n{name}:")
    print(f"  Tokens: {encoding.tokens}")
    print(f"  Token Count: {len(encoding.tokens)}")



BPE:
  Tokens: ['ĠÐ¼Ð¾Ð¸', 'ĠÐ´', 'ÑıÐ´', 'Ñı', 'ĠÑģÐ°Ð¼', 'ÑĭÑħ', 'ĠÑĩ', 'ÐµÑģÑĤ', 'Ð½ÑĭÑħ', 'ĠÐ¿ÑĢÐ°Ð²', 'Ð¸Ð»']
  Token Count: 11

WordPiece:
  Tokens: ['мои', 'д', '##я', '##дя', 'сам', '##ых', 'ч', '##ест', '##ных', 'пра', '##ви', '##л']
  Token Count: 12

Unigram:
  Tokens: ['▁мо', 'и', '▁д', 'я', 'д', 'я', '▁сам', 'ых', '▁', 'че', 'ст', 'ных', '▁прав', 'ил']
  Token Count: 14
